# Week 3 Day 5 - Capstone: Enterprise Assistant
Combining RAG, tools, memory, and routing into one working system.

**Course:** IPAM USL 3-Week Short Course: Advanced Artificial Intelligence  
**Facilitator:** Solomon Wilson | Deputy HOD Transport Planning & Operations | IT & Audit Supervisor, SLPTA  
**Mode:** Google Colab (zero-install)

## Learning objectives
    - Integrate retrieval, tool calling, and memory in a single pipeline.
- Route queries to the correct knowledge source or tool.
- Apply safety checks before returning answers.
- Evaluate the assistant against a set of test questions.

## Environment setup

Run this first in Google Colab. It installs the libraries used throughout the course.

In [ ]:
# Install dependencies required for the course notebooks.
# The -q flag keeps the installation output compact.
!pip install -q google-generativeai langchain langchain-google-genai chromadb pydantic pandas numpy scikit-learn

print("Dependencies installed. Restart the runtime if Colab asks you to.")

In [ ]:
"""
Utility functions used across notebooks.
These helpers are intentionally simple so students can read and modify them.
"""
import os
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def pretty_json(data: Any) -> str:
    """Return a nicely formatted JSON string for learning and debugging."""
    return json.dumps(data, indent=2, ensure_ascii=False)

def safe_preview(text: str, n: int = 300) -> str:
    """Return the first n characters of a string for quick inspection."""
    return text[:n] + ("..." if len(text) > n else "")

print("Helper functions loaded.")

## Full pipeline

In [ ]:
# --------------- Knowledge base ---------------
KB = {
    "hr": [
        "Leave must be approved by the line manager before travel.",
        "All staff must complete annual performance reviews."
    ],
    "finance": [
        "Procurement above Le 10 million requires board approval.",
        "Payment vouchers must carry authorising signatures."
    ],
    "operations": [
        "Vehicles must pass monthly safety checks.",
        "Route logs must be submitted within 48 hours of completion."
    ]
}

# --------------- Router ---------------
def route(question: str) -> str:
    """Return the best knowledge domain for the question."""
    q = question.lower()
    scores = {domain: sum(1 for w in terms if w in q)
              for domain, terms in {
                  "hr": ["leave", "staff", "performance", "recruit"],
                  "finance": ["payment", "procurement", "budget", "voucher"],
                  "operations": ["vehicle", "route", "dispatch", "log"]
              }.items()}
    return max(scores, key=scores.get)

print("Router test:")
for q in ["How do I apply for leave?", "What is the procurement limit?", "When are route logs due?"]:
    print(f"  '{q}' => {route(q)}")

In [ ]:
# --------------- Retriever ---------------
def retrieve(question: str, domain: str) -> str:
    """Return the most relevant KB entry from the chosen domain."""
    q_terms = set(question.lower().split())
    docs = KB.get(domain, [])
    ranked = sorted(docs, key=lambda d: len(q_terms & set(d.lower().split())), reverse=True)
    return ranked[0] if ranked else "No relevant document found."

# --------------- Safe answer ---------------
def answer(question: str) -> str:
    """Route, retrieve context, then generate a grounded response."""
    domain = route(question)
    context = retrieve(question, domain)
    if not context or context == "No relevant document found.":
        return "I do not have enough grounded context to answer safely."
    prompt = (
        f"Answer using only the context below.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}"
    )
    # Replace ask_llm(prompt) with a live call when running in Colab.
    print(f"[Domain: {domain}] Context: {context}")
    return f"(model response for: {question})"

# Test the full pipeline.
test_questions = [
    "What approvals do I need for a large purchase?",
    "When do I need to submit route logs?",
    "How does staff leave work?"
]
for q in test_questions:
    print("\nQ:", q)
    print("A:", answer(q))

### Final exercise
Extend the enterprise assistant with:
1. A conversation buffer (from Day 4) so it remembers the last three questions.
2. At least one new domain with two policy documents.
3. A printed evaluation table: question | domain | context found? | answered?

## Guided practice

## Course complete

You have built:
- LLM API calls with Gemini
- Reusable prompt templates in LangChain
- Validated structured outputs with Pydantic
- A retrieval-augmented generation pipeline
- Tool-calling and agent decision logic
- Conversation memory
- A full enterprise assistant prototype

The patterns in these notebooks scale directly to production systems.

## Submission checklist
- Run every code cell successfully.
- Add your own notes in markdown cells.
- Save a clean copy of the notebook after completing the exercises.